In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    SimpleRNN,
    LSTM,
    GRU,
    Bidirectional,
    Dense
)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)



I0000 00:00:1786424054.562798   38192 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786424056.132363   38192 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786424062.704104   38192 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
df = pd.read_csv("../Dataset/df_for_EDA.csv")

In [3]:
df = df.set_index("Datetime")

In [4]:
df = df.sort_index()

print(df.index.min())
print(df.index.max())
print(df.shape)

1998-04-01 01:00:00
2015-01-01 00:00:00
(145198, 14)


In [5]:
len(df) * 0.20

29039.600000000002

In [6]:
test_size = int(len(df) * 0.20)

test_df = df.iloc[-test_size:].copy()

remaining_df = df.iloc[:-test_size].copy()

In [7]:
60 * 24

1440

In [8]:
validation_hours = 60 * 24

val_df = remaining_df.iloc[-validation_hours:].copy()

train_df = remaining_df.iloc[:-validation_hours].copy()

In [9]:
print("Train:")
print(train_df.index.min(), "→", train_df.index.max())
print(train_df.shape)

print("\nValidation:")
print(val_df.index.min(), "→", val_df.index.max())
print(val_df.shape)

print("\nTest:")
print(test_df.index.min(), "→", test_df.index.max())
print(test_df.shape)

Train:
1998-04-01 01:00:00 → 2011-05-11 03:00:00
(114719, 14)

Validation:
2011-05-11 04:00:00 → 2011-07-10 03:00:00
(1440, 14)

Test:
2011-07-10 04:00:00 → 2015-01-01 00:00:00
(29039, 14)


In [10]:
scaler = StandardScaler()

train_scaled = scaler.fit_transform( train_df[["PJME_MW"]] )


val_scaled = scaler.transform( val_df[["PJME_MW"]] )


test_scaled = scaler.transform( test_df[["PJME_MW"]] )

In [11]:
train_scaled

array([[-1.61544069],
       [-1.73285452],
       [-1.77132563],
       ...,
       [ 0.73114322],
       [ 0.55817711],
       [ 0.16961888]], shape=(114719, 1))

In [12]:
test_scaled

array([[-1.14255379],
       [-1.49387197],
       [-1.6449865 ],
       ...,
       [ 0.1556154 ],
       [ 0.12730066],
       [ 0.0665163 ]], shape=(29039, 1))

In [13]:
val_scaled

array([[-0.21693885],
       [-0.75784267],
       [-1.00713547],
       ...,
       [-0.26341195],
       [-0.4983935 ],
       [-0.80985561]], shape=(1440, 1))

In [14]:
def create_sequences(data, sequence_length, forecast_horizon):

    X = []
    y = []

    for i in range(sequence_length, len(data) - forecast_horizon + 1):

        X.append(data[i-sequence_length:i])

        y.append(data[i:i+forecast_horizon])

    return np.array(X), np.array(y)

In [15]:
SEQUENCE_LENGTH = 24
FORECAST_HORIZON = 24


X_train, y_train = create_sequences(
    train_scaled,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [16]:
X_train

array([[[-1.61544069],
        [-1.73285452],
        [-1.77132563],
        ...,
        [-0.254025  ],
        [-0.59257078],
        [-0.95466088]],

       [[-1.73285452],
        [-1.77132563],
        [-1.76363141],
        ...,
        [-0.59257078],
        [-0.95466088],
        [-1.20949352]],

       [[-1.77132563],
        [-1.76363141],
        [-1.67699446],
        ...,
        [-0.95466088],
        [-1.20949352],
        [-1.33844868]],

       ...,

       [[ 0.29980512],
        [-0.03412412],
        [-0.40806333],
        ...,
        [ 1.17617704],
        [ 0.90672537],
        [ 0.71883247]],

       [[-0.03412412],
        [-0.40806333],
        [-0.4776191 ],
        ...,
        [ 0.90672537],
        [ 0.71883247],
        [ 0.55879264]],

       [[-0.40806333],
        [-0.4776191 ],
        [-0.7476863 ],
        ...,
        [ 0.71883247],
        [ 0.55879264],
        [ 0.18916221]]], shape=(114672, 24, 1))

In [17]:
y_train

array([[[-1.20949352],
        [-1.33844868],
        [-1.38322906],
        ...,
        [-0.34758674],
        [-0.74537803],
        [-1.1354751 ]],

       [[-1.33844868],
        [-1.38322906],
        [-1.37507318],
        ...,
        [-0.74537803],
        [-1.1354751 ],
        [-1.40738892]],

       [[-1.38322906],
        [-1.37507318],
        [-1.27843375],
        ...,
        [-1.1354751 ],
        [-1.40738892],
        [-1.55465633]],

       ...,

       [[ 0.55879264],
        [ 0.18916221],
        [-0.23278894],
        ...,
        [ 0.68851723],
        [ 0.49893159],
        [ 0.73114322]],

       [[ 0.18916221],
        [-0.23278894],
        [-0.634889  ],
        ...,
        [ 0.49893159],
        [ 0.73114322],
        [ 0.55817711]],

       [[-0.23278894],
        [-0.634889  ],
        [-0.90634116],
        ...,
        [ 0.73114322],
        [ 0.55817711],
        [ 0.16961888]]], shape=(114672, 24, 1))

In [18]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (114672, 24, 1)
y_train shape: (114672, 24, 1)


In [19]:
val_input = np.concatenate([
    train_scaled[-SEQUENCE_LENGTH:],
    val_scaled
])


X_val, y_val = create_sequences(
    val_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [20]:
val_input

array([[-0.23278894],
       [-0.634889  ],
       [-0.90634116],
       ...,
       [-0.26341195],
       [-0.4983935 ],
       [-0.80985561]], shape=(1464, 1))

In [21]:
X_val

array([[[-0.23278894],
        [-0.634889  ],
        [-0.90634116],
        ...,
        [ 0.73114322],
        [ 0.55817711],
        [ 0.16961888]],

       [[-0.634889  ],
        [-0.90634116],
        [-1.09161803],
        ...,
        [ 0.55817711],
        [ 0.16961888],
        [-0.21693885]],

       [[-0.90634116],
        [-1.09161803],
        [-1.21195567],
        ...,
        [ 0.16961888],
        [-0.21693885],
        [-0.75784267]],

       ...,

       [[-0.48623663],
        [-0.87017832],
        [-1.22072708],
        ...,
        [-0.26725906],
        [-0.09783229],
        [-0.19785717]],

       [[-0.87017832],
        [-1.22072708],
        [-1.50679827],
        ...,
        [-0.09783229],
        [-0.19785717],
        [-0.47392587]],

       [[-1.22072708],
        [-1.50679827],
        [-1.64437096],
        ...,
        [-0.19785717],
        [-0.47392587],
        [-0.82801398]]], shape=(1417, 24, 1))

In [22]:
y_val

array([[[-0.21693885],
        [-0.75784267],
        [-1.00713547],
        ...,
        [ 1.17525373],
        [ 0.95335236],
        [ 0.4830815 ]],

       [[-0.75784267],
        [-1.00713547],
        [-1.1770239 ],
        ...,
        [ 0.95335236],
        [ 0.4830815 ],
        [-0.0027317 ]],

       [[-1.00713547],
        [-1.1770239 ],
        [-1.28674351],
        ...,
        [ 0.4830815 ],
        [-0.0027317 ],
        [-0.58579986]],

       ...,

       [[-0.47392587],
        [-0.82801398],
        [-1.17748555],
        ...,
        [-0.22324811],
        [-0.0719797 ],
        [-0.26341195]],

       [[-0.82801398],
        [-1.17748555],
        [-1.49017875],
        ...,
        [-0.0719797 ],
        [-0.26341195],
        [-0.4983935 ]],

       [[-1.17748555],
        [-1.49017875],
        [-1.64052385],
        ...,
        [-0.26341195],
        [-0.4983935 ],
        [-0.80985561]]], shape=(1417, 24, 1))

In [23]:
test_input = np.concatenate([
    val_scaled[-SEQUENCE_LENGTH:],
    test_scaled
])


X_test, y_test = create_sequences(
    test_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [24]:
test_input

array([[-1.17748555],
       [-1.49017875],
       [-1.64052385],
       ...,
       [ 0.1556154 ],
       [ 0.12730066],
       [ 0.0665163 ]], shape=(29063, 1))

In [25]:
X_test

array([[[-1.17748555],
        [-1.49017875],
        [-1.64052385],
        ...,
        [-0.26341195],
        [-0.4983935 ],
        [-0.80985561]],

       [[-1.49017875],
        [-1.64052385],
        [-1.72362145],
        ...,
        [-0.4983935 ],
        [-0.80985561],
        [-1.14255379]],

       [[-1.64052385],
        [-1.72362145],
        [-1.74162593],
        ...,
        [-0.80985561],
        [-1.14255379],
        [-1.49387197]],

       ...,

       [[-0.70506031],
        [-0.75630383],
        [ 0.02173593],
        ...,
        [-0.26525856],
        [-0.33327549],
        [-0.41083325]],

       [[-0.75630383],
        [ 0.02173593],
        [ 0.01281063],
        ...,
        [-0.33327549],
        [-0.41083325],
        [-0.43637807]],

       [[ 0.02173593],
        [ 0.01281063],
        [ 0.01219509],
        ...,
        [-0.41083325],
        [-0.43637807],
        [-0.41868135]]], shape=(29016, 24, 1))

In [26]:
y_test

array([[[-1.14255379],
        [-1.49387197],
        [-1.6449865 ],
        ...,
        [-0.86186856],
        [-1.01005928],
        [-1.21349451]],

       [[-1.49387197],
        [-1.6449865 ],
        [-1.72192872],
        ...,
        [-1.01005928],
        [-1.21349451],
        [-1.4470911 ]],

       [[-1.6449865 ],
        [-1.72192872],
        [-1.74239535],
        ...,
        [-1.21349451],
        [-1.4470911 ],
        [-1.44170515]],

       ...,

       [[-0.43637807],
        [-0.41868135],
        [-0.84817284],
        ...,
        [ 0.19562535],
        [ 0.18639229],
        [ 0.1556154 ]],

       [[-0.41868135],
        [-0.84817284],
        [-0.7990837 ],
        ...,
        [ 0.18639229],
        [ 0.1556154 ],
        [ 0.12730066]],

       [[-0.84817284],
        [-0.7990837 ],
        [-0.73245174],
        ...,
        [ 0.1556154 ],
        [ 0.12730066],
        [ 0.0665163 ]]], shape=(29016, 24, 1))

In [27]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (114672, 24, 1)
y_train: (114672, 24, 1)
X_val: (1417, 24, 1)
y_val: (1417, 24, 1)
X_test: (29016, 24, 1)
y_test: (29016, 24, 1)


# Basic RNN

In [28]:
def build_rnn_model(sequence_length):

    model = Sequential([
        
        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        SimpleRNN(
            64,
            activation="tanh"
        ),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model

In [29]:
rnn_model = build_rnn_model(
    SEQUENCE_LENGTH
)

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

E0000 00:00:1786424160.698820   38192 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [30]:
history_rnn = rnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 22s 10ms/step - loss: 0.1612 - mae: 0.2912 - val_loss: 0.2142 - val_mae: 0.3364
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - loss: 0.1296 - mae: 0.2601 - val_loss: 0.2501 - val_mae: 0.3661
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.1238 - mae: 0.2529 - val_loss: 0.2257 - val_mae: 0.3475
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 36s 13ms/step - loss: 0.1193 - mae: 0.2472 - val_loss: 0.2178 - val_mae: 0.3380
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 28s 15ms/step - loss: 0.1162 - mae: 0.2434 - val_loss: 0.2303 - val_mae: 0.3499
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 37s 14ms/step - loss: 0.1134 - mae: 0.2405 - val_loss: 0.2085 - val_mae: 0.3290
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.1109 - mae: 0.2374 - val_loss: 0.2399 - val_mae: 0.3552
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.1087 - mae: 0.2350 - val_loss: 0.2230 - val_mae: 0.3377
Epoch 9/10
1792/1792 ━━━━

In [ ]:
history_rnn = rnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
    
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0840 - mae: 0.2117 - val_loss: 0.0637 - val_mae: 0.1853 - learning_rate: 0.0010
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - loss: 0.0839 - mae: 0.2118 - val_loss: 0.0491 - val_mae: 0.1619 - learning_rate: 0.0010
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - loss: 0.0829 - mae: 0.2105 - val_loss: 0.0617 - val_mae: 0.1848 - learning_rate: 0.0010
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.0823 - mae: 0.2099 - val_loss: 0.0478 - val_mae: 0.1597 - learning_rate: 0.0010
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - loss: 0.0812 - mae: 0.2084 - val_loss: 0.0513 - val_mae: 0.1667 - learning_rate: 0.0010
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - loss: 0.0804 - mae: 0.2072 - val_loss: 0.0471 - val_mae: 0.1594 - learning_rate: 0.0010
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - loss: 0.0799 - mae: 0.2065 - val_loss: 0.0464 - val_mae: 0.1602 - learnin

# LSTM

In [25]:
def build_lstm_model(sequence_length):

    model = Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        LSTM(64),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model

In [33]:
lstm_model = build_lstm_model(
    SEQUENCE_LENGTH
)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

history_lstm = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 13ms/step - loss: 0.1766 - mae: 0.3088 - val_loss: 0.0767 - val_mae: 0.2018
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - loss: 0.1092 - mae: 0.2440 - val_loss: 0.0606 - val_mae: 0.1772
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.1008 - mae: 0.2337 - val_loss: 0.0657 - val_mae: 0.1883
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0969 - mae: 0.2283 - val_loss: 0.0604 - val_mae: 0.1767
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.0939 - mae: 0.2242 - val_loss: 0.0562 - val_mae: 0.1718
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.0914 - mae: 0.2207 - val_loss: 0.0549 - val_mae: 0.1687
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - loss: 0.0891 - mae: 0.2175 - val_loss: 0.0542 - val_mae: 0.1693
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 25s 14ms/step - loss: 0.0876 - mae: 0.2154 - val_loss: 0.0586 - val_mae: 0.1754
Epoch 9/10
1792/1792 ━━━

# GRU

In [26]:
def build_gru_model(sequence_length):

    model = Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        GRU(64),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model

In [35]:
gru_model = build_gru_model(
    SEQUENCE_LENGTH
)

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_gru = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 33s 16ms/step - loss: 0.1789 - mae: 0.3077 - val_loss: 0.0730 - val_mae: 0.1962
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.1117 - mae: 0.2462 - val_loss: 0.0811 - val_mae: 0.2079
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 29s 16ms/step - loss: 0.1028 - mae: 0.2355 - val_loss: 0.0894 - val_mae: 0.2232
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0978 - mae: 0.2292 - val_loss: 0.0681 - val_mae: 0.1905
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 0.0942 - mae: 0.2245 - val_loss: 0.0704 - val_mae: 0.1943
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 41s 15ms/step - loss: 0.0912 - mae: 0.2206 - val_loss: 0.0607 - val_mae: 0.1807
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 0.0884 - mae: 0.2168 - val_loss: 0.0534 - val_mae: 0.1670
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 26s 15ms/step - loss: 0.0859 - mae: 0.2136 - val_loss: 0.0476 - val_mae: 0.1569
Epoch 9/10
1792/1792 ━━━

# LSTM + Bidirectional

In [ ]:
def build_bilstm_model(sequence_length):

    model = Sequential([

        tf.keras.layers.Input(
            shape=(sequence_length, 1)
        ),

        Bidirectional(
            LSTM(64)
            
        ),

        Dense(64, activation="relu"),

        Dense(24)
    ])

    return model



In [38]:
bilstm_model = build_bilstm_model(
    SEQUENCE_LENGTH
)

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_bilstm = bilstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 75s 37ms/step - loss: 0.1511 - mae: 0.2835 - val_loss: 0.0674 - val_mae: 0.1892
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 49s 27ms/step - loss: 0.1049 - mae: 0.2387 - val_loss: 0.0759 - val_mae: 0.2039
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - loss: 0.0978 - mae: 0.2294 - val_loss: 0.0597 - val_mae: 0.1741
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 52s 29ms/step - loss: 0.0933 - mae: 0.2233 - val_loss: 0.0579 - val_mae: 0.1750
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 44s 24ms/step - loss: 0.0899 - mae: 0.2187 - val_loss: 0.0630 - val_mae: 0.1819
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 40s 22ms/step - loss: 0.0867 - mae: 0.2142 - val_loss: 0.0603 - val_mae: 0.1767
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 44s 25ms/step - loss: 0.0837 - mae: 0.2101 - val_loss: 0.0546 - val_mae: 0.1680
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 47s 26ms/step - loss: 0.0809 - mae: 0.2062 - val_loss: 0.0502 - val_mae: 0.1590
Epoch 9/10
1792/1792 ━━━

In [39]:
rnn_pred = rnn_model.predict(
    X_test
)

lstm_pred = lstm_model.predict(
    X_test
)

gru_pred = gru_model.predict(
    X_test
)

bilstm_pred = bilstm_model.predict(
    X_test
)

907/907 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step


In [40]:
rnn_pred

array([[-0.98236305, -1.1773653 , -1.3379735 , ..., -0.05086365,
        -0.3281464 , -0.6497523 ],
       [-1.2789485 , -1.3627292 , -1.4237107 , ..., -0.3937418 ,
        -0.7459058 , -0.947224  ],
       [-1.5271153 , -1.5898683 , -1.6117347 , ..., -0.70814604,
        -1.0258298 , -1.2596714 ],
       ...,
       [ 1.2444727 ,  0.8547008 ,  0.584122  , ...,  2.1134279 ,
         2.014485  ,  1.8637037 ],
       [ 0.8579916 ,  0.50949216,  0.3693345 , ...,  2.0326931 ,
         1.8534814 ,  1.5468844 ],
       [ 0.5841448 ,  0.32907543,  0.2987256 , ...,  1.9806972 ,
         1.6389323 ,  1.2001114 ]], shape=(29016, 24), dtype=float32)

In [41]:
lstm_pred

array([[-1.0339231 , -1.2772336 , -1.436019  , ..., -0.04974754,
        -0.3624664 , -0.6767298 ],
       [-1.2767748 , -1.4241961 , -1.5378155 , ..., -0.34222573,
        -0.7226193 , -0.99429274],
       [-1.5397735 , -1.6436062 , -1.6772805 , ..., -0.73920304,
        -1.0781935 , -1.2977225 ],
       ...,
       [ 1.2708527 ,  0.79982126,  0.47288364, ...,  2.084054  ,
         1.9476794 ,  1.6866069 ],
       [ 0.86661685,  0.5300539 ,  0.32884455, ...,  2.0595398 ,
         1.7841105 ,  1.4589174 ],
       [ 0.58009523,  0.3989885 ,  0.32478458, ...,  1.8437864 ,
         1.4647771 ,  1.1074753 ]], shape=(29016, 24), dtype=float32)

In [42]:
gru_pred

array([[-1.0418394 , -1.240251  , -1.3790203 , ...,  0.04230755,
        -0.25595686, -0.6157472 ],
       [-1.2761693 , -1.4134693 , -1.5176489 , ..., -0.2962089 ,
        -0.66202915, -0.9760723 ],
       [-1.4770218 , -1.5842029 , -1.6263608 , ..., -0.65823865,
        -1.0153581 , -1.2890549 ],
       ...,
       [ 1.4874526 ,  1.1683657 ,  0.8905362 , ...,  2.2833655 ,
         2.1807811 ,  1.9652877 ],
       [ 1.3095893 ,  1.0279827 ,  0.77259773, ...,  2.100257  ,
         1.9594995 ,  1.6773474 ],
       [ 1.1330478 ,  0.8729358 ,  0.7200676 , ...,  1.8706336 ,
         1.5879427 ,  1.2296402 ]], shape=(29016, 24), dtype=float32)

In [43]:
bilstm_pred

array([[-1.0080543 , -1.1750776 , -1.307158  , ..., -0.03861562,
        -0.37374806, -0.7252225 ],
       [-1.2875327 , -1.3820951 , -1.4739033 , ..., -0.4009152 ,
        -0.68230283, -0.8928752 ],
       [-1.5442355 , -1.6039561 , -1.6016681 , ..., -0.86552817,
        -1.1030685 , -1.218047  ],
       ...,
       [ 1.414811  ,  0.9988717 ,  0.60385287, ...,  2.1057782 ,
         1.9298843 ,  1.7164263 ],
       [ 0.9204057 ,  0.60039264,  0.37882727, ...,  1.8782209 ,
         1.6182475 ,  1.2880441 ],
       [ 0.5800745 ,  0.45264882,  0.37300885, ...,  1.7966943 ,
         1.4428414 ,  0.9681557 ]], shape=(29016, 24), dtype=float32)

# ReScaling

In [ ]:
rnn_pred_original = scaler.inverse_transform(
    rnn_pred.reshape(-1, 1)
).reshape(rnn_pred.shape)

lstm_pred_original = scaler.inverse_transform(
    lstm_pred.reshape(-1, 1)
).reshape(lstm_pred.shape)

gru_pred_original = scaler.inverse_transform(
    gru_pred.reshape(-1, 1)
).reshape(gru_pred.shape)

bilstm_pred_original = scaler.inverse_transform(
    bilstm_pred.reshape(-1, 1)
).reshape(bilstm_pred.shape)

In [46]:
rnn_pred_original

array([[25927.555, 24670.445, 23635.062, ..., 31932.59 , 30145.05 ,
        28071.773],
       [24015.574, 23475.473, 23082.346, ..., 29722.18 , 27451.906,
        26154.082],
       [22415.734, 22011.188, 21870.223, ..., 27695.33 , 25647.34 ,
        24139.848],
       ...,
       [40283.152, 37770.434, 36026.113, ..., 45884.992, 45247.14 ,
        44275.11 ],
       [37791.65 , 35545.   , 34641.453, ..., 45364.523, 44209.21 ,
        42232.69 ],
       [36026.258, 34381.918, 34186.266, ..., 45029.324, 42826.09 ,
        39997.17 ]], shape=(29016, 24), dtype=float32)

In [47]:
lstm_pred_original

array([[25595.164, 24026.63 , 23003.   , ..., 31939.785, 29923.8  ,
        27897.86 ],
       [24029.59 , 23079.217, 22346.754, ..., 30054.285, 27602.027,
        25850.646],
       [22334.133, 21664.76 , 21447.674, ..., 27495.117, 25309.77 ,
        23894.547],
       ...,
       [40453.215, 37416.65 , 35309.   , ..., 45695.625, 44816.47 ,
        43133.43 ],
       [37847.254, 35677.555, 34380.43 , ..., 45537.594, 43762.   ,
        41665.6  ],
       [36000.152, 34832.625, 34354.258, ..., 44146.71 , 41703.375,
        39399.98 ]], shape=(29016, 24), dtype=float32)

In [41]:
y_test_original = scaler.inverse_transform(
    y_test.reshape(-1, 1)
).reshape(y_test.shape)

# Error Calculation

In [48]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [51]:
deep_results = {}

deep_results["RNN"] = evaluate_deep_model(
    y_test_original,
    rnn_pred_original
)

deep_results["LSTM"] = evaluate_deep_model(
    y_test_original,
    lstm_pred_original
)

deep_results["GRU"] = evaluate_deep_model(
    y_test_original,
    gru_pred_original
)

deep_results["Bi-LSTM"] = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original
)

In [52]:
deep_results

{'RNN': {'MAE': 1582.632554135508,
  'MSE': 4720266.125651733,
  'RMSE': np.float64(2172.6173445067893),
  'MAPE': 5.1174328903127275,
  'R2': 0.8892683575234632,
  'Bias': np.float64(321.9579765278455)},
 'LSTM': {'MAE': 1576.632464123938,
  'MSE': 4697914.9737056345,
  'RMSE': np.float64(2167.4674100677116),
  'MAPE': 5.072953668259601,
  'R2': 0.8897926880803325,
  'Bias': np.float64(137.18421656269655)},
 'GRU': {'MAE': 1554.157947776927,
  'MSE': 4582343.315316676,
  'RMSE': np.float64(2140.6408655626183),
  'MAPE': 5.0580674280512286,
  'R2': 0.8925038571577708,
  'Bias': np.float64(461.5139159595317)},
 'Bi-LSTM': {'MAE': 1480.638330575112,
  'MSE': 4262538.484320015,
  'RMSE': np.float64(2064.5916023078303),
  'MAPE': 4.787187228935753,
  'R2': 0.9000060854782773,
  'Bias': np.float64(122.36444579808877)}}

In [53]:
deep_results_df = pd.DataFrame(
    deep_results
).T

deep_results_df

,MAE,MSE,RMSE,MAPE,R2,Bias
RNN,1582.632554,4.720266e+06,2172.617345,5.117433,0.889268,321.957977
LSTM,1576.632464,4.697915e+06,2167.467410,5.072954,0.889793,137.184217
GRU,1554.157948,4.582343e+06,2140.640866,5.058067,0.892504,461.513916
Bi-LSTM,1480.638331,4.262538e+06,2064.591602,4.787187,0.900006,122.364446


# 48

In [15]:
SEQUENCE_LENGTH = 48
FORECAST_HORIZON = 24


X_train, y_train = create_sequences(
    train_scaled,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [16]:
X_train

array([[[-0.78296491],
        [-1.0265028 ],
        [-1.14067088],
        ...,
        [ 0.34087713],
        [ 0.02815586],
        [-0.37577577]],

       [[-1.0265028 ],
        [-1.14067088],
        [-1.16952314],
        ...,
        [ 0.02815586],
        [-0.37577577],
        [-0.79397841]],

       [[-1.14067088],
        [-1.16952314],
        [-1.10390751],
        ...,
        [-0.37577577],
        [-0.79397841],
        [-1.01828145]],

       ...,

       [[-0.52469065],
        [-0.80824942],
        [-0.64397768],
        ...,
        [ 0.65468423],
        [ 0.55059892],
        [ 0.30799176]],

       [[-0.80824942],
        [-0.64397768],
        [-0.76171351],
        ...,
        [ 0.55059892],
        [ 0.30799176],
        [-0.05715996]],

       [[-0.64397768],
        [-0.76171351],
        [-0.8144542 ],
        ...,
        [ 0.30799176],
        [-0.05715996],
        [-0.39671692]]], shape=(114648, 48, 1))

In [17]:
y_train

array([[[-0.79397841],
        [-1.01828145],
        [-1.14067088],
        ...,
        [-0.07996255],
        [-0.3484747 ],
        [-0.71796977]],

       [[-1.01828145],
        [-1.14067088],
        [-1.20070219],
        ...,
        [-0.3484747 ],
        [-0.71796977],
        [-0.82081411]],

       [[-1.14067088],
        [-1.20070219],
        [-1.21031961],
        ...,
        [-0.71796977],
        [-0.82081411],
        [-1.00959475]],

       ...,

       [[-0.05715996],
        [-0.39671692],
        [-1.38575996],
        ...,
        [-0.06569154],
        [-0.15659167],
        [-0.34940542]],

       [[-0.39671692],
        [-1.38575996],
        [-1.52847006],
        ...,
        [-0.15659167],
        [-0.34940542],
        [-0.60736944]],

       [[-1.38575996],
        [-1.52847006],
        [-1.61114885],
        ...,
        [-0.34940542],
        [-0.60736944],
        [-0.85974915]]], shape=(114648, 24, 1))

In [18]:
val_input = np.concatenate([
    train_scaled[-SEQUENCE_LENGTH:],
    val_scaled
])


X_val, y_val = create_sequences(
    val_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [19]:
val_input

array([[-1.03069103],
       [-1.1124391 ],
       [-1.14547959],
       ...,
       [ 0.05793884],
       [-0.28720243],
       [-0.71502249]], shape=(1488, 1))

In [20]:
test_input = np.concatenate([
    val_scaled[-SEQUENCE_LENGTH:],
    test_scaled
])


X_test, y_test = create_sequences(
    test_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

# Basic RNN with 48

In [23]:
rnn_model = build_rnn_model(
    SEQUENCE_LENGTH
)

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

history_rnn = rnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

E0000 00:00:1786343704.975355   77511 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - loss: 0.1413 - mae: 0.2727 - val_loss: 0.0674 - val_mae: 0.1889
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0978 - mae: 0.2310 - val_loss: 0.0710 - val_mae: 0.1939
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0898 - mae: 0.2209 - val_loss: 0.0638 - val_mae: 0.1850
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0839 - mae: 0.2132 - val_loss: 0.0624 - val_mae: 0.1859
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0797 - mae: 0.2076 - val_loss: 0.0557 - val_mae: 0.1793
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0756 - mae: 0.2017 - val_loss: 0.0477 - val_mae: 0.1613
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - loss: 0.0731 - mae: 0.1981 - val_loss: 0.0412 - val_mae: 0.1496
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 18s 10ms/step - loss: 0.0702 - mae: 0.1938 - val_loss: 0.0395 - val_mae: 0.1480
Epoch 9/10
1792/1792 ━━━

# LSTM 48

In [28]:
lstm_model = build_lstm_model(
    SEQUENCE_LENGTH
)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

history_lstm = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 49s 26ms/step - loss: 0.1487 - mae: 0.2795 - val_loss: 0.0574 - val_mae: 0.1764
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 45s 25ms/step - loss: 0.0919 - mae: 0.2244 - val_loss: 0.0558 - val_mae: 0.1693
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 54s 30ms/step - loss: 0.0853 - mae: 0.2151 - val_loss: 0.0510 - val_mae: 0.1605
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 74s 25ms/step - loss: 0.0805 - mae: 0.2080 - val_loss: 0.0572 - val_mae: 0.1735
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 45s 25ms/step - loss: 0.0761 - mae: 0.2017 - val_loss: 0.0559 - val_mae: 0.1726
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 45s 25ms/step - loss: 0.0722 - mae: 0.1961 - val_loss: 0.0581 - val_mae: 0.1766
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 45s 25ms/step - loss: 0.0685 - mae: 0.1905 - val_loss: 0.0475 - val_mae: 0.1593
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 44s 25ms/step - loss: 0.0654 - mae: 0.1858 - val_loss: 0.0515 - val_mae: 0.1647
Epoch 9/10
1792/1792 ━━━

# GRU 48

In [30]:
gru_model = build_gru_model(
    SEQUENCE_LENGTH
)

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_gru = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 67s 36ms/step - loss: 0.1762 - mae: 0.3043 - val_loss: 0.0889 - val_mae: 0.2195
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 50s 28ms/step - loss: 0.0966 - mae: 0.2297 - val_loss: 0.0688 - val_mae: 0.1883
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 48s 27ms/step - loss: 0.0878 - mae: 0.2182 - val_loss: 0.0551 - val_mae: 0.1697
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 49s 27ms/step - loss: 0.0822 - mae: 0.2103 - val_loss: 0.0555 - val_mae: 0.1703
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 50s 28ms/step - loss: 0.0753 - mae: 0.2008 - val_loss: 0.0438 - val_mae: 0.1525
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 48s 27ms/step - loss: 0.0689 - mae: 0.1917 - val_loss: 0.0433 - val_mae: 0.1557
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 47s 26ms/step - loss: 0.0639 - mae: 0.1838 - val_loss: 0.0366 - val_mae: 0.1406
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 48s 27ms/step - loss: 0.0606 - mae: 0.1787 - val_loss: 0.0303 - val_mae: 0.1279
Epoch 9/10
1792/1792 ━━━

# BI-directional 48

In [32]:
bilstm_model = build_bilstm_model(
    SEQUENCE_LENGTH
)

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_bilstm = bilstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 73s 38ms/step - loss: 0.1448 - mae: 0.2766 - val_loss: 0.0730 - val_mae: 0.1944
Epoch 2/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 61s 34ms/step - loss: 0.0909 - mae: 0.2224 - val_loss: 0.0558 - val_mae: 0.1673
Epoch 3/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 74s 41ms/step - loss: 0.0839 - mae: 0.2124 - val_loss: 0.0491 - val_mae: 0.1566
Epoch 4/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 85s 43ms/step - loss: 0.0789 - mae: 0.2049 - val_loss: 0.0492 - val_mae: 0.1606
Epoch 5/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 93s 52ms/step - loss: 0.0744 - mae: 0.1983 - val_loss: 0.0479 - val_mae: 0.1592
Epoch 6/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 135s 48ms/step - loss: 0.0699 - mae: 0.1917 - val_loss: 0.0466 - val_mae: 0.1546
Epoch 7/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 90s 50ms/step - loss: 0.0659 - mae: 0.1858 - val_loss: 0.0506 - val_mae: 0.1704
Epoch 8/10
1792/1792 ━━━━━━━━━━━━━━━━━━━━ 71s 40ms/step - loss: 0.0608 - mae: 0.1779 - val_loss: 0.0414 - val_mae: 0.1511
Epoch 9/10
1792/1792 ━━

In [33]:
rnn_pred_48 = rnn_model.predict(
    X_test
)

lstm_pred_48 = lstm_model.predict(
    X_test
)

gru_pred_48 = gru_model.predict(
    X_test
)

bilstm_pred_48 = bilstm_model.predict(
    X_test
)

907/907 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step


In [34]:
rnn_pred_48

array([[-1.1338766 , -1.4063342 , -1.5485914 , ..., -0.12454487,
        -0.36477873, -0.7650793 ],
       [-1.3496863 , -1.5299572 , -1.6504005 , ..., -0.38159397,
        -0.7048263 , -1.00547   ],
       [-1.5577059 , -1.6316425 , -1.6729473 , ..., -0.71637195,
        -0.954239  , -1.193554  ],
       ...,
       [ 1.2671335 ,  0.80376786,  0.35947847, ...,  1.7468112 ,
         1.6962069 ,  1.5040201 ],
       [ 0.8914399 ,  0.49581736,  0.20081982, ...,  1.6729282 ,
         1.6488023 ,  1.426319  ],
       [ 0.66886985,  0.39156944,  0.25951177, ...,  1.6016462 ,
         1.3823464 ,  1.1047132 ]], shape=(29016, 24), dtype=float32)

In [35]:
lstm_pred_48

array([[-1.0944514 , -1.3411666 , -1.5138623 , ...,  0.00999829,
        -0.31314173, -0.7607372 ],
       [-1.3615496 , -1.4881904 , -1.581114  , ..., -0.32142335,
        -0.68975973, -1.1061368 ],
       [-1.5582464 , -1.6649615 , -1.6947505 , ..., -0.78517425,
        -1.1174779 , -1.3663414 ],
       ...,
       [ 1.3506933 ,  0.9357484 ,  0.47922987, ...,  1.8892035 ,
         1.7533131 ,  1.5328245 ],
       [ 0.89097184,  0.49322873,  0.2204106 , ...,  1.8490598 ,
         1.5618778 ,  1.2133474 ],
       [ 0.5786575 ,  0.32856685,  0.24612783, ...,  1.6732422 ,
         1.2810189 ,  0.8639652 ]], shape=(29016, 24), dtype=float32)

In [36]:
gru_pred_48

array([[-1.0726472 , -1.2223101 , -1.3495208 , ...,  0.00652675,
        -0.31327167, -0.6961062 ],
       [-1.1985048 , -1.2762004 , -1.4144152 , ..., -0.27175915,
        -0.69370496, -1.0028195 ],
       [-1.4891076 , -1.5480293 , -1.5850261 , ..., -0.6911859 ,
        -1.0303735 , -1.331902  ],
       ...,
       [ 1.4055156 ,  0.8608271 ,  0.37960488, ...,  1.8427252 ,
         1.7578504 ,  1.4979895 ],
       [ 0.8383707 ,  0.40135038,  0.15099671, ...,  1.7830478 ,
         1.5287237 ,  1.1501292 ],
       [ 0.5407156 ,  0.3668779 ,  0.32793745, ...,  1.616958  ,
         1.2865815 ,  0.8408713 ]], shape=(29016, 24), dtype=float32)

In [37]:
bilstm_pred_48

array([[-1.0750061 , -1.3241466 , -1.4864169 , ...,  0.06780823,
        -0.2730954 , -0.7569097 ],
       [-1.3054295 , -1.4514495 , -1.5736252 , ..., -0.18412021,
        -0.6245644 , -1.066421  ],
       [-1.5230124 , -1.6380337 , -1.6447271 , ..., -0.6487246 ,
        -1.0433415 , -1.3845522 ],
       ...,
       [ 1.3203472 ,  0.88156813,  0.47752488, ...,  1.9800658 ,
         1.8506893 ,  1.5896847 ],
       [ 0.8629273 ,  0.548776  ,  0.33367354, ...,  1.8526306 ,
         1.61164   ,  1.2619531 ],
       [ 0.6986089 ,  0.48645577,  0.42911315, ...,  1.6289659 ,
         1.3042771 ,  0.9337573 ]], shape=(29016, 24), dtype=float32)

In [42]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [45]:
rnn_pred_original_48 = scaler.inverse_transform(
    rnn_pred_48.reshape(-1, 1)
).reshape(rnn_pred_48.shape)

lstm_pred_original_48    = scaler.inverse_transform(
    lstm_pred_48.reshape(-1, 1)
).reshape(lstm_pred_48.shape)

gru_pred_original_48 = scaler.inverse_transform(
    gru_pred_48.reshape(-1, 1)
).reshape(gru_pred_48.shape)

bilstm_pred_original_48 = scaler.inverse_transform(
    bilstm_pred_48.reshape(-1, 1)
).reshape(bilstm_pred_48.shape)

In [46]:
deep_results = {}

deep_results["RNN"] = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_48
)

deep_results["LSTM"] = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_48
)

deep_results["GRU"] = evaluate_deep_model(
    y_test_original,
    gru_pred_original_48
)

deep_results["Bi-LSTM"] = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_48
)

In [47]:
deep_results

{'RNN': {'MAE': 1402.8555002856722,
  'MSE': 3725879.183714366,
  'RMSE': np.float64(1930.2536578684071),
  'MAPE': 4.503843801509725,
  'R2': 0.9125954531589328,
  'Bias': np.float64(-94.71735237850095)},
 'LSTM': {'MAE': 1411.5486115290064,
  'MSE': 3852194.3719179933,
  'RMSE': np.float64(1962.700785121867),
  'MAPE': 4.536547166797631,
  'R2': 0.909632254074985,
  'Bias': np.float64(70.17388189887421)},
 'GRU': {'MAE': 1342.3335059269114,
  'MSE': 3525377.203512186,
  'RMSE': np.float64(1877.5987866187456),
  'MAPE': 4.344329677679464,
  'R2': 0.9172989832135057,
  'Bias': np.float64(131.98055823569504)},
 'Bi-LSTM': {'MAE': 1403.3674982626774,
  'MSE': 3771535.9106302387,
  'RMSE': np.float64(1942.044260728946),
  'MAPE': 4.587355281825901,
  'R2': 0.9115244024539687,
  'Bias': np.float64(483.9693117747356)}}

In [48]:
deep_results_df_48 = pd.DataFrame(
    deep_results
).T

deep_results_df_48

,MAE,MSE,RMSE,MAPE,R2,Bias
RNN,1402.855500,3.725879e+06,1930.253658,4.503844,0.912595,-94.717352
LSTM,1411.548612,3.852194e+06,1962.700785,4.536547,0.909632,70.173882
GRU,1342.333506,3.525377e+06,1877.598787,4.344330,0.917299,131.980558
Bi-LSTM,1403.367498,3.771536e+06,1942.044261,4.587355,0.911524,483.969312


# 168

In [ ]:
SEQUENCE_LENGTH = 168
FORECAST_HORIZON = 24


X_train, y_train = create_sequences(
    train_scaled,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [51]:
val_input = np.concatenate([
    train_scaled[-SEQUENCE_LENGTH:],
    val_scaled
])


X_val, y_val = create_sequences(
    val_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

In [52]:
test_input = np.concatenate([
    val_scaled[-SEQUENCE_LENGTH:],
    test_scaled
])


X_test, y_test = create_sequences(
    test_input,
    SEQUENCE_LENGTH,
    FORECAST_HORIZON
)

# Basic RNN 168

In [53]:
rnn_model = build_rnn_model(
    SEQUENCE_LENGTH
)

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

history_rnn = rnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 124s 67ms/step - loss: 0.1336 - mae: 0.2642 - val_loss: 0.0622 - val_mae: 0.1794
Epoch 2/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 127s 71ms/step - loss: 0.0938 - mae: 0.2260 - val_loss: 0.0631 - val_mae: 0.1871
Epoch 3/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 155s 79ms/step - loss: 0.0863 - mae: 0.2160 - val_loss: 0.0769 - val_mae: 0.2045
Epoch 4/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 89s 49ms/step - loss: 0.0895 - mae: 0.2166 - val_loss: 0.1513 - val_mae: 0.3041
Epoch 5/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 87s 49ms/step - loss: 0.1732 - mae: 0.3091 - val_loss: 0.0963 - val_mae: 0.2249
Epoch 6/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 136s 45ms/step - loss: 0.1192 - mae: 0.2572 - val_loss: 0.0703 - val_mae: 0.1947
Epoch 7/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 83s 46ms/step - loss: 0.1004 - mae: 0.2357 - val_loss: 0.0832 - val_mae: 0.2153
Epoch 8/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 100s 56ms/step - loss: 0.0987 - mae: 0.2322 - val_loss: 0.0855 - val_mae: 0.2218
Epoch 9/10
1790/179

# LSTM 168

In [54]:
lstm_model = build_lstm_model(
    SEQUENCE_LENGTH
)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

history_lstm = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
)

Epoch 1/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 236s 130ms/step - loss: 0.1351 - mae: 0.2664 - val_loss: 0.0617 - val_mae: 0.1897
Epoch 2/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 263s 130ms/step - loss: 0.0793 - mae: 0.2081 - val_loss: 0.0500 - val_mae: 0.1710
Epoch 3/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 216s 120ms/step - loss: 0.0705 - mae: 0.1957 - val_loss: 0.0374 - val_mae: 0.1465
Epoch 4/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 190s 106ms/step - loss: 0.0647 - mae: 0.1872 - val_loss: 0.0383 - val_mae: 0.1486
Epoch 5/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 253s 141ms/step - loss: 0.0605 - mae: 0.1806 - val_loss: 0.0349 - val_mae: 0.1403
Epoch 6/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 263s 147ms/step - loss: 0.0569 - mae: 0.1745 - val_loss: 0.0518 - val_mae: 0.1749
Epoch 7/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 153s 85ms/step - loss: 0.0544 - mae: 0.1701 - val_loss: 0.0397 - val_mae: 0.1529
Epoch 8/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 154s 86ms/step - loss: 0.0519 - mae: 0.1658 - val_loss: 0.0409 - val_mae: 0.1592
Epoch 9/10

# GRU 168

In [55]:
gru_model = build_gru_model(
    SEQUENCE_LENGTH
)

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_gru = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 207s 114ms/step - loss: 0.1677 - mae: 0.2973 - val_loss: 0.0682 - val_mae: 0.1878
Epoch 2/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 274s 121ms/step - loss: 0.0849 - mae: 0.2167 - val_loss: 0.0530 - val_mae: 0.1721
Epoch 3/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 227s 127ms/step - loss: 0.0729 - mae: 0.2000 - val_loss: 0.0503 - val_mae: 0.1721
Epoch 4/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 231s 110ms/step - loss: 0.0667 - mae: 0.1907 - val_loss: 0.0347 - val_mae: 0.1386
Epoch 5/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 183s 102ms/step - loss: 0.0613 - mae: 0.1821 - val_loss: 0.0401 - val_mae: 0.1512
Epoch 6/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 199s 111ms/step - loss: 0.0568 - mae: 0.1749 - val_loss: 0.0326 - val_mae: 0.1381
Epoch 7/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 215s 120ms/step - loss: 0.0571 - mae: 0.1739 - val_loss: 0.0309 - val_mae: 0.1331
Epoch 8/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 205s 115ms/step - loss: 0.0518 - mae: 0.1658 - val_loss: 0.0286 - val_mae: 0.1279
Epoch 9/

# Bidirectional LSTM 168

In [56]:
bilstm_model = build_bilstm_model(
    SEQUENCE_LENGTH
)

bilstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)


history_bilstm = bilstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64
)

Epoch 1/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 200s 109ms/step - loss: 0.1181 - mae: 0.2495 - val_loss: 0.0453 - val_mae: 0.1629
Epoch 2/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 204s 110ms/step - loss: 0.0686 - mae: 0.1943 - val_loss: 0.0483 - val_mae: 0.1717
Epoch 3/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 191s 107ms/step - loss: 0.0605 - mae: 0.1810 - val_loss: 0.0385 - val_mae: 0.1487
Epoch 4/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 240s 134ms/step - loss: 0.0558 - mae: 0.1732 - val_loss: 0.0291 - val_mae: 0.1265
Epoch 5/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 232s 117ms/step - loss: 0.0529 - mae: 0.1679 - val_loss: 0.0392 - val_mae: 0.1525
Epoch 6/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 270s 122ms/step - loss: 0.0499 - mae: 0.1624 - val_loss: 0.0268 - val_mae: 0.1202
Epoch 7/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 234s 131ms/step - loss: 0.0477 - mae: 0.1585 - val_loss: 0.0279 - val_mae: 0.1237
Epoch 8/10
1790/1790 ━━━━━━━━━━━━━━━━━━━━ 219s 122ms/step - loss: 0.0462 - mae: 0.1554 - val_loss: 0.0265 - val_mae: 0.1193
Epoch 9/

In [57]:
rnn_pred_168 = rnn_model.predict(
    X_test
)

lstm_pred_168 = lstm_model.predict(
    X_test
)

gru_pred_168 = gru_model.predict(
    X_test
)

bilstm_pred_168 = bilstm_model.predict(
    X_test
)

907/907 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 18s 20ms/step
907/907 ━━━━━━━━━━━━━━━━━━━━ 26s 28ms/step


In [58]:
def evaluate_deep_model(actual, predicted):

    actual_flat = actual.reshape(-1)
    predicted_flat = predicted.reshape(-1)

    mae = mean_absolute_error(
        actual_flat,
        predicted_flat
    )

    mse = mean_squared_error(
        actual_flat,
        predicted_flat
    )

    rmse = np.sqrt(mse)

    mape = mean_absolute_percentage_error(
        actual_flat,
        predicted_flat
    ) * 100

    r2 = r2_score(
        actual_flat,
        predicted_flat
    )

    bias = np.mean(
        predicted_flat - actual_flat
    )

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,
        "Bias": bias
    }

In [59]:
rnn_pred_original_168 = scaler.inverse_transform(
    rnn_pred_168.reshape(-1, 1)
).reshape(rnn_pred_168.shape)

lstm_pred_original_168 = scaler.inverse_transform(
    lstm_pred_168.reshape(-1, 1)
).reshape(lstm_pred_168.shape)

gru_pred_original_168 = scaler.inverse_transform(
    gru_pred_168.reshape(-1, 1)
).reshape(gru_pred_168.shape)

bilstm_pred_original_168 = scaler.inverse_transform(
    bilstm_pred_168.reshape(-1, 1)
).reshape(bilstm_pred_168.shape)

In [60]:
deep_results = {}

deep_results["RNN"] = evaluate_deep_model(
    y_test_original,
    rnn_pred_original_168
)

deep_results["LSTM"] = evaluate_deep_model(
    y_test_original,
    lstm_pred_original_168
)

deep_results["GRU"] = evaluate_deep_model(
    y_test_original,
    gru_pred_original_168
)

deep_results["Bi-LSTM"] = evaluate_deep_model(
    y_test_original,
    bilstm_pred_original_168
)

In [61]:
deep_results

{'RNN': {'MAE': 1698.9008009252977,
  'MSE': 5257774.550526484,
  'RMSE': np.float64(2292.9837658663187),
  'MAPE': 5.523372234784886,
  'R2': 0.8766590704309607,
  'Bias': np.float64(455.5622663151724)},
 'LSTM': {'MAE': 1879.7937438364645,
  'MSE': 6367207.364543782,
  'RMSE': np.float64(2523.3325909486807),
  'MAPE': 6.117971971328903,
  'R2': 0.8506331400187132,
  'Bias': np.float64(382.4745696609198)},
 'GRU': {'MAE': 1233.2888963627647,
  'MSE': 2955252.121529323,
  'RMSE': np.float64(1719.0846754972029),
  'MAPE': 3.9799966285603907,
  'R2': 0.9306734169984898,
  'Bias': np.float64(95.65914989072121)},
 'Bi-LSTM': {'MAE': 1210.2719647442852,
  'MSE': 2936766.0301443017,
  'RMSE': np.float64(1713.6995157098872),
  'MAPE': 3.8769999088506757,
  'R2': 0.9311070779844479,
  'Bias': np.float64(13.133739568547131)}}

In [62]:
deep_results_df_48 = pd.DataFrame(
    deep_results
).T

deep_results_df_48


,MAE,MSE,RMSE,MAPE,R2,Bias
RNN,1698.900801,5.257775e+06,2292.983766,5.523372,0.876659,455.562266
LSTM,1879.793744,6.367207e+06,2523.332591,6.117972,0.850633,382.474570
GRU,1233.288896,2.955252e+06,1719.084675,3.979997,0.930673,95.659150
Bi-LSTM,1210.271965,2.936766e+06,1713.699516,3.877000,0.931107,13.133740


In [64]:
bilstm_model.save(
    "../Models/bilstm_model_168.keras", 
)

In [65]:
sequence_results = [

    # Sequence Length = 24
    {"Sequence": 24, "Model": "RNN",
     "MAE": 1582.632554, "MSE": 4720266.126,
     "RMSE": 2172.617345, "MAPE": 5.117433,
     "R2": 0.889268, "Bias": 321.957977},

    {"Sequence": 24, "Model": "LSTM",
     "MAE": 1576.632464, "MSE": 4697914.974,
     "RMSE": 2167.467410, "MAPE": 5.072954,
     "R2": 0.889793, "Bias": 137.184217},

    {"Sequence": 24, "Model": "GRU",
     "MAE": 1554.157948, "MSE": 4582343.315,
     "RMSE": 2140.640866, "MAPE": 5.058067,
     "R2": 0.892504, "Bias": 461.513916},

    {"Sequence": 24, "Model": "Bi-LSTM",
     "MAE": 1480.638331, "MSE": 4262538.484,
     "RMSE": 2064.591602, "MAPE": 4.787187,
     "R2": 0.900006, "Bias": 122.364446},


    # Sequence Length = 48
    {"Sequence": 48, "Model": "RNN",
     "MAE": 1402.855500, "MSE": 3725879.184,
     "RMSE": 1930.253658, "MAPE": 4.503844,
     "R2": 0.912595, "Bias": -94.717352},

    {"Sequence": 48, "Model": "LSTM",
     "MAE": 1411.548612, "MSE": 3852194.372,
     "RMSE": 1962.700785, "MAPE": 4.536547,
     "R2": 0.909632, "Bias": 70.173882},

    {"Sequence": 48, "Model": "GRU",
     "MAE": 1342.333506, "MSE": 3525377.204,
     "RMSE": 1877.598787, "MAPE": 4.344330,
     "R2": 0.917299, "Bias": 131.980558},

    {"Sequence": 48, "Model": "Bi-LSTM",
     "MAE": 1403.367498, "MSE": 3771535.911,
     "RMSE": 1942.044261, "MAPE": 4.587355,
     "R2": 0.911524, "Bias": 483.969312},


    # Sequence Length = 168
    {"Sequence": 168, "Model": "RNN",
     "MAE": 1698.900801, "MSE": 5257774.551,
     "RMSE": 2292.983765, "MAPE": 5.523372,
     "R2": 0.876659, "Bias": 455.562266},

    {"Sequence": 168, "Model": "LSTM",
     "MAE": 1879.793744, "MSE": 6367207.365,
     "RMSE": 2523.332591, "MAPE": 6.117972,
     "R2": 0.850633, "Bias": 382.474570},

    {"Sequence": 168, "Model": "GRU",
     "MAE": 1233.288896, "MSE": 2955252.122,
     "RMSE": 1719.084675, "MAPE": 3.979997,
     "R2": 0.930673, "Bias": 95.659150},

    {"Sequence": 168, "Model": "Bi-LSTM",
     "MAE": 1210.271965, "MSE": 2936766.030,
     "RMSE": 1713.699516, "MAPE": 3.877000,
     "R2": 0.931107, "Bias": 13.133740}
]

sequence_df = pd.DataFrame(sequence_results)

sequence_df

,Sequence,Model,MAE,MSE,RMSE,MAPE,R2,Bias
0,24,RNN,1582.632554,4720266.126,2172.617345,5.117433,0.889268,321.957977
1,24,LSTM,1576.632464,4697914.974,2167.467410,5.072954,0.889793,137.184217
2,24,GRU,1554.157948,4582343.315,2140.640866,5.058067,0.892504,461.513916
3,24,Bi-LSTM,1480.638331,4262538.484,2064.591602,4.787187,0.900006,122.364446
4,48,RNN,1402.855500,3725879.184,1930.253658,4.503844,0.912595,-94.717352
5,48,LSTM,1411.548612,3852194.372,1962.700785,4.536547,0.909632,70.173882
6,48,GRU,1342.333506,3525377.204,1877.598787,4.344330,0.917299,131.980558
7,48,Bi-LSTM,1403.367498,3771535.911,1942.044261,4.587355,0.911524,483.969312
8,168,RNN,1698.900801,5257774.551,2292.983765,5.523372,0.876659,455.562266
9,168,LSTM,1879.793744,6367207.365,2523.332591,6.117972,0.850633,382.474570


In [66]:
sequence_df.sort_values(by=["R2"], ascending=False)

,Sequence,Model,MAE,MSE,RMSE,MAPE,R2,Bias
11,168,Bi-LSTM,1210.271965,2936766.030,1713.699516,3.877000,0.931107,13.133740
10,168,GRU,1233.288896,2955252.122,1719.084675,3.979997,0.930673,95.659150
6,48,GRU,1342.333506,3525377.204,1877.598787,4.344330,0.917299,131.980558
4,48,RNN,1402.855500,3725879.184,1930.253658,4.503844,0.912595,-94.717352
7,48,Bi-LSTM,1403.367498,3771535.911,1942.044261,4.587355,0.911524,483.969312
5,48,LSTM,1411.548612,3852194.372,1962.700785,4.536547,0.909632,70.173882
3,24,Bi-LSTM,1480.638331,4262538.484,2064.591602,4.787187,0.900006,122.364446
2,24,GRU,1554.157948,4582343.315,2140.640866,5.058067,0.892504,461.513916
1,24,LSTM,1576.632464,4697914.974,2167.467410,5.072954,0.889793,137.184217
0,24,RNN,1582.632554,4720266.126,2172.617345,5.117433,0.889268,321.957977
